In [1]:
!curl -I https://google.com 2>&1 | head -5

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0   220    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0




In [2]:
!pip install -q transformers accelerate bitsandbytes datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.2 MB/s eta 0:00:00


In [3]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
print("Token loaded:", os.environ["HF_TOKEN"][:8], "...")

Token loaded: hf_biSNj ...


In [4]:
%%writefile phase1_baselines.py
"""
phase1_baselines.py  v2
========================
Fragility Under Constraint — Phase 1: Baseline Evaluation
Target hardware: Kaggle T4 (15 GB VRAM)

What this script does
---------------------
Runs all 5 models x 3 tiers x 3 domains on CLEAN data (no shift).
Collects per-cell: Accuracy, F1, ECE proxy, inference latency.
Writes results to baseline_results.csv and baseline_results.json.

Models (5)
----------
Gemma 3 4B, Mistral 7B, Llama 3.1 8B, Qwen 2.5 3B, SmolLM 1.7B

Tiers (3)
---------
Tier 0: FP16  |  Tier 1: INT8  |  Tier 2: INT4

Domains (3)
-----------
Domain A - General NLP       : SST-2 sentiment (binary classification)
Domain B - Financial NLP     : Twitter Financial Sentiment (3-class)
Domain C - Structured Reason : SVAMP math word problems (exact match)

Samples per cell: 50
Total cells     : 5 x 3 x 3 = 45
Estimated time  : 3-5 hours on T4

Usage
-----
python phase1_baselines.py --hf-token YOUR_TOKEN
python phase1_baselines.py --resume --hf-token YOUR_TOKEN
"""

import argparse
import csv
import gc
import json
import logging
import os
import re
import sys
import time
import traceback
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# ---------------------------------------------------------------------------
# Warning / log noise suppression — must be here, in the subprocess itself.
# ---------------------------------------------------------------------------
warnings.filterwarnings(
    "ignore", message=".*inputs will be cast from torch.bfloat16 to float16.*"
)
warnings.filterwarnings("once")

logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["BITSANDBYTES_NOWELCOME"] = "1"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("phase1")

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
SAMPLES_PER_CELL = 50
VRAM_LIMIT_GB    = 14.0
CHECKPOINT_FILE  = "phase1_checkpoint.json"
RESULTS_CSV      = "baseline_results.csv"
RESULTS_JSON     = "baseline_results.json"

MAX_CELL_SECONDS = 1800  # 30 min hard budget per cell

EAGER_ATTN_MODELS = {"qwen25"}
BFLOAT16_MODELS   = {"gemma3"}
CHAT_TEMPLATE_MODELS = {"gemma3", "mistral7b", "llama31", "qwen25", "smollm"}
EXTRA_EOS_TOKENS  = {"gemma3": [1, 107]}

MODEL_REGISTRY: Dict[str, Dict[str, Any]] = {
    "gemma3":    {"name": "Gemma 3 4B",   "hf_id": "google/gemma-3-4b-it",                 "params_b": 4.0},
    "mistral7b": {"name": "Mistral 7B",   "hf_id": "mistralai/Mistral-7B-Instruct-v0.3",    "params_b": 7.0},
    "llama31":   {"name": "Llama 3.1 8B", "hf_id": "meta-llama/Meta-Llama-3.1-8B-Instruct", "params_b": 8.0},
    "qwen25":    {"name": "Qwen 2.5 3B",  "hf_id": "Qwen/Qwen2.5-3B-Instruct",              "params_b": 3.0},
    "smollm":    {"name": "SmolLM 1.7B",  "hf_id": "HuggingFaceTB/SmolLM2-1.7B-Instruct",   "params_b": 1.7},
}

TIERS: Dict[str, Dict[str, Any]] = {
    "fp16": {"label": "Tier 0 — FP16", "load_in_8bit": False, "load_in_4bit": False},
    "int8": {"label": "Tier 1 — INT8", "load_in_8bit": True,  "load_in_4bit": False},
    "int4": {"label": "Tier 2 — INT4", "load_in_8bit": False, "load_in_4bit": True},
}

DOMAIN_REGISTRY: Dict[str, Dict[str, Any]] = {
    "general_nlp": {
        "name": "General NLP",
        "task": "sentiment",
        "dataset_id": "stanfordnlp/sst2",
        "dataset_split": "validation",
        "text_field": "sentence",
        "label_field": "label",
        "label_map": {0: "negative", 1: "positive"},
        "prompt_template": (
            "Classify the sentiment of the following text.\n"
            "Reply with exactly one word: 'positive' or 'negative'.\n"
            "Text: {text}\nAnswer:"
        ),
        "valid_answers": ["positive", "negative"],
        "metric": "accuracy",
    },
    "financial_nlp": {
        "name": "Financial NLP",
        "task": "sentiment",
        "dataset_id": "zeroshot/twitter-financial-news-sentiment",
        "dataset_split": "train",
        "text_field": "text",
        "label_field": "label",
        "label_map": {0: "negative", 1: "positive", 2: "neutral"},
        "prompt_template": (
            "Classify the sentiment of this financial news as exactly "
            "'positive', 'negative', or 'neutral'.\n"
            "Text: {text}\nAnswer:"
        ),
        "valid_answers": ["positive", "negative", "neutral"],
        "metric": "accuracy",
    },
    "structured_reasoning": {
        "name": "Structured Reasoning",
        "task": "math",
        "dataset_id": "ChilleD/SVAMP",
        "dataset_split": "test",
        "text_field": "question_concat",
        "label_field": "Answer",
        "label_map": {},
        "prompt_template": (
            "Solve this math problem step by step, then write the final answer "
            "on the last line as: Answer: <number>\n"
            "Problem: {text}"
        ),
        "valid_answers": [],
        "metric": "exact_match",
    },
}


@dataclass
class CellResult:
    model_key: str
    model_name: str
    tier: str
    domain: str
    task: str
    n_samples: int
    accuracy: float
    f1_macro: float
    ece_proxy: float
    latency_ms_mean: float
    latency_ms_std: float
    status: str          # "ok" | "error" | "timeout_partial"
    error_msg: str = ""
    timestamp: str = ""


def gb(b: int) -> float:
    return round(b / (1024 ** 3), 2)


def flush_gpu(torch_mod) -> None:
    """
    Aggressive VRAM release between model loads. The extra synchronize +
    sleep is deliberate — back-to-back model loads on a single T4 have
    shown intermittent CUDA OOM errors (llama31/int8, mistral7b/int4)
    that plain empty_cache() didn't reliably prevent, likely allocator
    fragmentation that needs a moment to actually release.
    """
    gc.collect()
    torch_mod.cuda.empty_cache()
    torch_mod.cuda.synchronize()
    torch_mod.cuda.reset_peak_memory_stats()
    time.sleep(2)


def cell_id(model_key: str, tier: str, domain: str) -> str:
    return f"{model_key}__{tier}__{domain}"


def load_checkpoint(path: str) -> set:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        completed = set(data.get("completed", []))
        log.info(f"Checkpoint loaded — {len(completed)} cells already done.")
        return completed
    return set()


def save_checkpoint(path: str, completed: set) -> None:
    with open(path, "w") as f:
        json.dump({"completed": list(completed)}, f)


def append_result_csv(result: CellResult, path: str) -> None:
    file_exists = os.path.exists(path)
    with open(path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(asdict(result).keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(asdict(result))


def import_dependencies() -> Dict[str, Any]:
    missing = []
    deps: Dict[str, Any] = {}
    try:
        import torch
        deps["torch"] = torch
    except ImportError:
        missing.append("torch")
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        deps["AutoTokenizer"]        = AutoTokenizer
        deps["AutoModelForCausalLM"] = AutoModelForCausalLM
        deps["BitsAndBytesConfig"]   = BitsAndBytesConfig
    except ImportError:
        missing.append("transformers")
    try:
        from datasets import load_dataset
        deps["load_dataset"] = load_dataset
    except ImportError:
        missing.append("datasets")
    try:
        import numpy as np
        deps["np"] = np
    except ImportError:
        missing.append("numpy")
    try:
        from sklearn.metrics import f1_score
        deps["f1_score"] = f1_score
    except ImportError:
        missing.append("scikit-learn")
    if missing:
        raise ImportError(
            f"\n\nMissing: {' '.join(missing)}\n"
            f"Run: pip install {' '.join(missing)} accelerate bitsandbytes\n"
        )
    return deps


def load_model_and_tokenizer(
    model_key: str,
    tier_key: str,
    deps: Dict[str, Any],
    hf_token: Optional[str] = None,
) -> Tuple[Any, Any]:
    torch     = deps["torch"]
    AutoTok   = deps["AutoTokenizer"]
    AutoModel = deps["AutoModelForCausalLM"]
    BnBConfig = deps["BitsAndBytesConfig"]

    model_cfg = MODEL_REGISTRY[model_key]
    tier_cfg  = TIERS[tier_key]

    log.info(f"Loading {model_cfg['name']} @ {tier_cfg['label']} ...")
    flush_gpu(torch)

    tok_kwargs: Dict[str, Any] = {"trust_remote_code": True}
    if hf_token:
        tok_kwargs["token"] = hf_token

    tokenizer = AutoTok.from_pretrained(model_cfg["hf_id"], **tok_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_cfg = None
    if tier_cfg["load_in_8bit"]:
        bnb_cfg = BnBConfig(load_in_8bit=True)
    elif tier_cfg["load_in_4bit"]:
        bnb_cfg = BnBConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

    dtype = torch.bfloat16 if model_key in BFLOAT16_MODELS else torch.float16

    load_kwargs: Dict[str, Any] = {
        "device_map": "auto",
        "trust_remote_code": True,
    }
    if bnb_cfg is not None:
        load_kwargs["quantization_config"] = bnb_cfg
        # Deliberately not forcing torch_dtype for quantized tiers.
    else:
        load_kwargs["dtype"] = dtype

    if model_key in EAGER_ATTN_MODELS:
        load_kwargs["attn_implementation"] = "eager"
    if hf_token:
        load_kwargs["token"] = hf_token

    model = AutoModel.from_pretrained(model_cfg["hf_id"], **load_kwargs)
    model.eval()

    vram_used = gb(torch.cuda.memory_allocated())
    log.info(f"  Loaded. VRAM: {vram_used} GB | dtype: {dtype}")
    return model, tokenizer


def normalize_number(s: str) -> str:
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(round(f, 4))
    except (ValueError, TypeError):
        return s.strip()


def extract_answer(output_text: str, valid_answers: List[str], task: str) -> str:
    text = output_text.strip().lower()
    if task == "math":
        cot_match = re.search(r"answer[:\s]+(-?\d+(?:\.\d+)?)", text)
        if cot_match:
            return normalize_number(cot_match.group(1))
        nums = re.findall(r"-?\d+(?:\.\d+)?", text)
        return normalize_number(nums[-1]) if nums else ""
    for ans in valid_answers:
        if ans.lower() in text:
            return ans.lower()
    return text.split()[0] if text.split() else ""


def run_inference(
    model, tokenizer, prompt: str, valid_answers: List[str], task: str,
    deps: Dict[str, Any], model_key: str = "", max_new_tokens: int = 16,
) -> Tuple[str, float, float]:
    torch = deps["torch"]

    if model_key in CHAT_TEMPLATE_MODELS and hasattr(tokenizer, "apply_chat_template"):
        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True,
        )
    else:
        text = prompt

    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=512, padding=False,
    ).to("cuda")

    eos_ids = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
    eos_ids = list(set(eos_ids + EXTRA_EOS_TOKENS.get(model_key, []))) or None

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id or 0, eos_token_id=eos_ids,
        )
    latency_ms = (time.time() - t0) * 1000

    input_len = inputs["input_ids"].shape[1]
    decoded   = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    pred = extract_answer(decoded, valid_answers, task)

    if task != "math" and valid_answers:
        try:
            with torch.no_grad():
                logits = model(
                    input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
                ).logits[:, -1, :]
            if logits.isnan().any():
                confidence = 1.0 / max(len(valid_answers), 1)
            else:
                probs = torch.softmax(logits, dim=-1)
                ans_probs = []
                for ans in valid_answers:
                    tok_id = tokenizer.encode(f" {ans}", add_special_tokens=False)
                    ans_probs.append(probs[0, tok_id[0]].item() if tok_id else 0.0)
                confidence = max(ans_probs) / (sum(ans_probs) + 1e-9)
        except Exception:
            confidence = 1.0 / max(len(valid_answers), 1)
    else:
        confidence = 1.0

    return pred, confidence, latency_ms


def evaluate_cell(
    model_key: str, tier_key: str, domain_key: str, model, tokenizer,
    deps: Dict[str, Any], n_samples: int = SAMPLES_PER_CELL,
) -> CellResult:
    np       = deps["np"]
    f1_score = deps["f1_score"]
    load_dataset = deps["load_dataset"]

    domain_cfg = DOMAIN_REGISTRY[domain_key]
    model_cfg  = MODEL_REGISTRY[model_key]
    task       = domain_cfg["task"]

    log.info(
        f"  Evaluating {model_cfg['name']} | {TIERS[tier_key]['label']} | "
        f"{domain_cfg['name']} | {n_samples} samples ..."
    )

    try:
        ds_kwargs: Dict[str, Any] = {
            "split": domain_cfg["dataset_split"],
            "streaming": True,
        }
        ds = load_dataset(domain_cfg["dataset_id"], **ds_kwargs)
        if domain_key == "financial_nlp":
            ds = ds.shuffle(buffer_size=10000, seed=42)
        samples = list(ds.take(n_samples))

        preds, labels, confs, lats = [], [], [], []
        cell_start = time.time()
        timed_out = False

        for i, sample in enumerate(samples):
            elapsed = time.time() - cell_start
            if elapsed > MAX_CELL_SECONDS:
                log.warning(
                    f"  ⚠ Cell exceeded {MAX_CELL_SECONDS}s budget after "
                    f"{i}/{n_samples} samples — truncating."
                )
                timed_out = True
                break

            text  = str(sample[domain_cfg["text_field"]])
            label = sample[domain_cfg["label_field"]]

            if domain_cfg["label_map"] and isinstance(label, int):
                label_str = domain_cfg["label_map"].get(label, str(label))
            elif task == "math":
                label_str = normalize_number(str(label))
            else:
                label_str = str(label).strip().lower()

            prompt = domain_cfg["prompt_template"].format(text=text[:400])

            pred, conf, lat = run_inference(
                model, tokenizer, prompt, domain_cfg["valid_answers"], task, deps,
                model_key=model_key, max_new_tokens=150 if task == "math" else 16,
            )

            preds.append(pred)
            labels.append(label_str)
            confs.append(conf)
            lats.append(lat)

            if (i + 1) % 10 == 0:
                running_acc = sum(p == l for p, l in zip(preds, labels)) / len(preds)
                log.info(f"    [{i+1}/{n_samples}] running acc={running_acc:.3f}")

        correct = [p == l for p, l in zip(preds, labels)]
        accuracy = float(np.mean(correct)) if correct else 0.0

        unique_labels = list(set(labels))
        if len(unique_labels) > 1:
            try:
                f1 = float(f1_score(labels, preds, average="macro", zero_division=0))
            except Exception:
                f1 = accuracy
        else:
            f1 = accuracy

        ece = float(np.mean([abs(c - int(ok)) for c, ok in zip(confs, correct)])) if correct else 0.0
        lat_arr = np.array(lats) if lats else np.array([0.0])
        status = "timeout_partial" if timed_out else "ok"

        log.info(
            f"  Done ({status}) — acc={accuracy:.3f} f1={f1:.3f} "
            f"ece={ece:.3f} lat={float(np.mean(lat_arr)):.1f}ms n={len(preds)}/{n_samples}"
        )

        return CellResult(
            model_key=model_key, model_name=model_cfg["name"], tier=tier_key,
            domain=domain_key, task=task, n_samples=len(preds),
            accuracy=round(accuracy, 4), f1_macro=round(f1, 4), ece_proxy=round(ece, 4),
            latency_ms_mean=round(float(np.mean(lat_arr)), 2),
            latency_ms_std=round(float(np.std(lat_arr)), 2),
            status=status, timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"),
        )

    except Exception as exc:
        log.error(f"  Cell failed: {exc}")
        log.debug(traceback.format_exc())
        return CellResult(
            model_key=model_key, model_name=model_cfg["name"], tier=tier_key,
            domain=domain_key, task=task, n_samples=0, accuracy=0.0, f1_macro=0.0,
            ece_proxy=0.0, latency_ms_mean=0.0, latency_ms_std=0.0,
            status="error", error_msg=str(exc)[:200],
            timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"),
        )


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Phase 1 baseline evaluation")
    p.add_argument("--model",   choices=list(MODEL_REGISTRY.keys()) + ["all"], default="all")
    p.add_argument("--tiers",   nargs="+", choices=list(TIERS.keys()), default=["fp16","int8","int4"])
    p.add_argument("--domains", nargs="+", choices=list(DOMAIN_REGISTRY.keys()), default=list(DOMAIN_REGISTRY.keys()))
    p.add_argument("--samples", type=int, default=SAMPLES_PER_CELL)
    p.add_argument("--resume",  action="store_true")
    p.add_argument("--hf-token", type=str, default=os.environ.get("HF_TOKEN"))
    p.add_argument("--output-dir", type=str, default="/kaggle/working")
    return p.parse_args()


def main() -> None:
    args = parse_args()

    log.info("=" * 60)
    log.info("Fragility Under Constraint — Phase 1: Baselines")
    log.info("=" * 60)

    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = str(out_dir / CHECKPOINT_FILE)
    csv_path        = str(out_dir / RESULTS_CSV)
    json_path       = str(out_dir / RESULTS_JSON)

    try:
        deps = import_dependencies()
    except ImportError as e:
        log.error(str(e))
        sys.exit(1)

    torch = deps["torch"]
    if not torch.cuda.is_available():
        log.error("CUDA not available. Run on a GPU instance.")
        sys.exit(1)

    log.info(f"GPU: {torch.cuda.get_device_name(0)}")
    log.info(f"VRAM: {gb(torch.cuda.get_device_properties(0).total_memory)} GB")
    log.info(f"Per-cell time budget: {MAX_CELL_SECONDS}s")

    model_keys  = list(MODEL_REGISTRY.keys()) if args.model == "all" else [args.model]
    tier_keys   = args.tiers
    domain_keys = args.domains

    total_cells = len(model_keys) * len(tier_keys) * len(domain_keys)
    log.info(f"\nPlan: {len(model_keys)} models x {len(tier_keys)} tiers x {len(domain_keys)} domains = {total_cells} cells")
    log.info(f"Samples per cell: {args.samples}")

    completed = load_checkpoint(checkpoint_path) if args.resume else set()

    all_results: List[CellResult] = []
    cell_num = 0

    for model_key in model_keys:
        for tier_key in tier_keys:
            pending_domains = [
                d for d in domain_keys
                if cell_id(model_key, tier_key, d) not in completed
            ]
            if not pending_domains:
                log.info(f"Skipping {MODEL_REGISTRY[model_key]['name']} @ {TIERS[tier_key]['label']} — all done.")
                continue

            try:
                model, tokenizer = load_model_and_tokenizer(
                    model_key, tier_key, deps, hf_token=args.hf_token
                )
            except Exception as exc:
                log.error(f"Failed to load {MODEL_REGISTRY[model_key]['name']} @ {TIERS[tier_key]['label']}: {exc}")
                for domain_key in pending_domains:
                    cell_num += 1
                    result = CellResult(
                        model_key=model_key, model_name=MODEL_REGISTRY[model_key]["name"],
                        tier=tier_key, domain=domain_key, task=DOMAIN_REGISTRY[domain_key]["task"],
                        n_samples=0, accuracy=0.0, f1_macro=0.0, ece_proxy=0.0,
                        latency_ms_mean=0.0, latency_ms_std=0.0, status="error",
                        error_msg=f"Model load failed: {str(exc)[:150]}",
                        timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"),
                    )
                    all_results.append(result)
                    append_result_csv(result, csv_path)
                # still flush before trying the next model+tier
                flush_gpu(torch)
                continue

            for domain_key in pending_domains:
                cell_num += 1
                cid = cell_id(model_key, tier_key, domain_key)
                log.info(
                    f"\n[Cell {cell_num}/{total_cells}] "
                    f"{MODEL_REGISTRY[model_key]['name']} | {TIERS[tier_key]['label']} | "
                    f"{DOMAIN_REGISTRY[domain_key]['name']}"
                )

                result = evaluate_cell(
                    model_key, tier_key, domain_key, model, tokenizer, deps,
                    n_samples=args.samples,
                )
                all_results.append(result)
                append_result_csv(result, csv_path)
                completed.add(cid)
                save_checkpoint(checkpoint_path, completed)

                log.info(
                    f"  ✓ Cell saved. acc={result.accuracy:.3f} f1={result.f1_macro:.3f} "
                    f"ece={result.ece_proxy:.3f} lat={result.latency_ms_mean:.1f}ms status={result.status}"
                )

            log.info(f"Unloading {MODEL_REGISTRY[model_key]['name']} @ {TIERS[tier_key]['label']} ...")
            del model, tokenizer
            flush_gpu(torch)

    with open(json_path, "w") as f:
        json.dump([asdict(r) for r in all_results], f, indent=2)

    log.info("\n" + "=" * 60)
    log.info("PHASE 1 COMPLETE — BASELINE RESULTS SUMMARY")
    log.info("=" * 60)
    log.info(f"{'Model':20s} {'Tier':6s} {'Domain':22s} {'Acc':6s} {'F1':6s} {'ECE':6s} {'Lat(ms)':8s} {'Status':16s}")
    log.info("-" * 90)
    for r in all_results:
        log.info(
            f"{r.model_name:20s} {r.tier:6s} {r.domain:22s} "
            f"{r.accuracy:6.3f} {r.f1_macro:6.3f} {r.ece_proxy:6.3f} "
            f"{r.latency_ms_mean:8.1f} {r.status:16s}"
        )

    ok_count      = sum(1 for r in all_results if r.status == "ok")
    err_count     = sum(1 for r in all_results if r.status == "error")
    timeout_count = sum(1 for r in all_results if r.status == "timeout_partial")
    log.info(f"\nCells completed: {ok_count} ok, {err_count} errors, {timeout_count} timeout_partial")
    log.info(f"Results saved to: {csv_path}")
    log.info(f"Full JSON saved to: {json_path}")
    log.info("=" * 60)


if __name__ == "__main__":
    main()

Writing phase1_baselines.py


In [5]:
import os
for f in ["baseline_results.csv", "baseline_result.csv", "phase1_checkpoint.json",
          "baseline_results.json"]:
    path = f"/kaggle/working/{f}"
    if os.path.exists(path):
        os.remove(path)
print("Clean. Starting fresh Phase 1 run.")

Clean. Starting fresh Phase 1 run.


In [6]:
!python phase1_baselines.py --hf-token {os.environ["HF_TOKEN"]}

08:33:00  INFO      ============================================================
08:33:00  INFO      Fragility Under Constraint — Phase 1: Baselines
08:33:00  INFO      ============================================================
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
08:33:38  INFO      NumExpr defaulting to 4 threads.
08:33:40  INFO      TensorFlow version 2.20.0 available.
08:33:40  INFO      JAX version 0.7.2 available.
08:33:42  INFO      GPU: Tesla T4
08:33:42  INFO      VRAM: 14.56 GB
08:33:42  INFO      Per-cell time budget: 1800s
08:33:42  INFO      
Plan: 5 models x 3 tiers x 3 domains = 45 cells
08:33:42  INFO      Samples per cell: 50
08:33:42  INFO      Loading Gemma 3 4B @ Tier 0 — FP16 ...
08:33:45  INFO      HTTP Request: HEAD https://huggingface.co/google/gemma-3-4b-it/resolve/main/config.json

In [7]:
import pandas as pd
df = pd.read_csv("/kaggle/working/baseline_results.csv")
print("Total rows:", len(df), "(expect 45)")
print("Any zero-accuracy rows?", (df["accuracy"] == 0.0).any())
print("Status counts:", dict(df["status"].value_counts()))
print("\nFinancial NLP check (SmolLM should be ~0.60, not 0.0):")
print(df[(df.domain=="financial_nlp")][["model_key","tier","accuracy"]].to_string(index=False))

Total rows: 45 (expect 45)
Any zero-accuracy rows? False
Status counts: {'ok': np.int64(45)}

Financial NLP check (SmolLM should be ~0.60, not 0.0):
model_key tier  accuracy
   gemma3 fp16      0.58
   gemma3 int8      0.58
   gemma3 int4      0.56
mistral7b fp16      0.68
mistral7b int8      0.68
mistral7b int4      0.80
  llama31 fp16      0.62
  llama31 int8      0.66
  llama31 int4      0.62
   qwen25 fp16      0.40
   qwen25 int8      0.40
   qwen25 int4      0.56
   smollm fp16      0.60
   smollm int8      0.60
   smollm int4      0.60
